# MS-Dialog Phase 1 — Multi-Turn Experiment

Three-round clarifying-question pipeline on 100 MS-Dialog tech-support cases.

**Pipeline per record:**
1. Model sees `category` + `title` + `original_question` → preliminary solution + CQ1 + confidence
2. User simulator answers CQ1 → model gives updated solution + CQ2 + confidence
3. User simulator answers CQ2 → model gives updated solution + CQ3 + confidence
4. User simulator answers CQ3 → model gives final solution + final confidence

**Output:** one row per case with solutions and CQs at all 4 checkpoints

**CQ classification** is done separately via `run_msdialog_judge.py`

In [ ]:
import sys
from pathlib import Path

ROOT       = Path("../../").resolve()
sys.path.insert(0, str(ROOT))

from config import SIMULATOR_MODEL_ID, MSDIALOG_GEMINI_MODEL_ID, N_CQ_TURNS

DATASET      = "ms-dialog"
DATASETS_DIR = ROOT / "datasets" / DATASET
PROMPTS_DIR  = ROOT / "prompts"  / DATASET
MODEL_ID     = MSDIALOG_GEMINI_MODEL_ID
OUTPUTS_DIR  = ROOT / "outputs" / DATASET / MODEL_ID

CASES_PATH        = DATASETS_DIR / "msdialog_100.jsonl"
INSTRUCTION_FILE  = PROMPTS_DIR  / "phase1_instruction.txt"
CONTINUATION_FILE = PROMPTS_DIR  / "phase1_continuation_instruction.txt"
OUTPUT_CSV        = OUTPUTS_DIR  / "phase1_multiturn_results.csv"

REQUEST_INTERVAL = 1.0

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Specialist : {MODEL_ID}")
print(f"Simulator  : {SIMULATOR_MODEL_ID}")
print(f"N_CQ_TURNS : {N_CQ_TURNS}")
print(f"Cases      : {CASES_PATH}")
print(f"Output CSV : {OUTPUT_CSV}")

In [ ]:
import importlib
import json
import logging

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv, parse_json_response
from src.providers import GeminiProvider
from src.pipeline import (
    MsDialogMultiTurnPhase1Pipeline, UserSimulator,
    MSDIALOG_TURN_0_SCHEMA, MSDIALOG_CONTINUATION_SCHEMA, MSDIALOG_FINAL_SCHEMA,
    _MSDIALOG_FINAL_INSTRUCTION, _format_problem,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

## Smoke Test

In [ ]:
provider     = GeminiProvider(model_id=MODEL_ID)
sim_provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)

resp = provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp and "SMOKE" in resp.upper(), f"Specialist smoke test failed: {resp!r}"
print(f"Specialist smoke test PASSED: {resp.strip()}")

resp2 = sim_provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp2 and "SMOKE" in resp2.upper(), f"Simulator smoke test failed: {resp2!r}"
print(f"Simulator  smoke test PASSED: {resp2.strip()}")

## Load Dataset

In [ ]:
with open(CASES_PATH, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} records")

from collections import Counter
cats = Counter(r["category"] for r in records)
print(f"Categories: {len(cats)} unique")

## Dry Run — Single Record
Verify the full three-turn flow before running everything.

In [ ]:
simulator        = UserSimulator(sim_provider)
instruction      = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()
continuation_ins = CONTINUATION_FILE.read_text(encoding="utf-8").strip()

test = records[0]
print(f"Testing on: {test['id']} | {test['category']} | {test['title']}")
print(f"OQ: {test['original_question'][:200]}")
print()

history = []  # [(cq, sim_response), ...]

# Turn 0
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=_format_problem(test["title"], test["category"], test["original_question"]),
    temperature=0.0, max_tokens=5000, expect_json=MSDIALOG_TURN_0_SCHEMA,
)
p0 = parse_json_response(raw_0)
print(f"=== TURN 0 ===\n{json.dumps(p0, indent=2)}")
history.append((p0["clarifying_question"], None))

# CQ rounds
for turn_idx in range(1, N_CQ_TURNS + 1):
    cq = history[-1][0]
    sim = simulator.answer(cq, test["simulator_context"])
    history[-1] = (cq, sim)
    print(f"\n=== USER RESPONSE {turn_idx} ===\n{sim}")

    history_text = "\n\n".join(
        f"Clarifying question {i}: {q}\nUser's answer: {a}"
        for i, (q, a) in enumerate(history, 1)
    )
    base_msg = (
        f"{_format_problem(test['title'], test['category'], test['original_question'])}\n\n"
        f"{history_text}"
    )

    if turn_idx < N_CQ_TURNS:
        raw = provider.call(
            system_instruction=continuation_ins,
            user_message=base_msg,
            temperature=0.0, max_tokens=5000, expect_json=MSDIALOG_CONTINUATION_SCHEMA,
        )
        p = parse_json_response(raw)
        print(f"\n=== TURN {turn_idx} (continuation) ===\n{json.dumps(p, indent=2)}")
        history.append((p["clarifying_question"], None))
    else:
        raw = provider.call(
            system_instruction=_MSDIALOG_FINAL_INSTRUCTION,
            user_message=base_msg,
            temperature=0.0, max_tokens=5000, expect_json=MSDIALOG_FINAL_SCHEMA,
        )
        p = parse_json_response(raw)
        print(f"\n=== TURN {turn_idx} (FINAL) ===\n{json.dumps(p, indent=2)}")

print(f"\n--- ACCEPTED ANSWER ---\n{test['accepted_answer'][:400]}")

## Run Full Experiment
100 cases × 3 CQ rounds. Resumes automatically if interrupted.

In [ ]:
pipeline = MsDialogMultiTurnPhase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    continuation_instruction_file=CONTINUATION_FILE,
    output_csv=OUTPUT_CSV,
    n_turns=N_CQ_TURNS,
    request_interval=REQUEST_INTERVAL,
    simulator_provider=sim_provider,
)

pipeline.run(records)

## Results Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(OUTPUT_CSV)
valid = df[~df["was_blocked"]]

print(f"Total: {len(df)} | Blocked: {df['was_blocked'].sum()} | Valid: {len(valid)}")

checkpoints = [
    ("Preliminary (Turn 0)", "preliminary_confidence"),
    ("After CQ1",            "confidence_1"),
    ("After CQ2",            "confidence_2"),
    ("Final (After CQ3)",    "final_confidence"),
]

print(f"\n{'Checkpoint':<25} {'Mean Conf':>10}")
print("-" * 40)
for label, col in checkpoints:
    print(f"{label:<25} {valid[col].mean():>10.1f}")

print("\nSample CQ1s generated:")
for _, row in valid.head(5).iterrows():
    print(f"  [{row['id']}] {row['cq_1'][:100]}")

print("\nNote: CQ classification (EPISTEMIC/ALEATORIC) → run run_msdialog_judge.py")